<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-09-ai-agents/lesson-9.5-multi-agent-crewai-and-agno/notebooks/exercises-9_5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 9.5: Multi-Agent with CrewAI & Agno

*Module 9 · AI Agents & Autonomous Systems*

> One agent is powerful. A TEAM of agents is transformative. A researcher finds information, an analyst interprets it, a writer produces the report. Each agent has a role, a backstory, specific tools, and they collaborate through delegation. This lesson builds multi-agent teams with CrewAI (role-based crews) and Agno (lightweight teams), then compares both frameworks.

`CrewAI` · `Agno` · `Role Specialization` · `Delegation` · `75 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-9.5.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — CrewAI — Role-Based Agent Teams — `01_crewai_research.py`
2. Step 2 — Agno — Lightweight Multi-Agent Teams — `02_agno_team.py`
3. Step 3 — Multi-Agent From Scratch — Understand the Pattern — `03_multi_agent_scratch.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai crewai


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GEMINI_API_KEY=sk-...
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY_HERE")
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · CrewAI — Role-Based Agent Teams

Define agents with roles, goals, backstories. Assign tasks. Let the crew collaborate.

**`01_crewai_research.py`** · _Block 1 of 3_

CREWAI — Build a research team — pip install crewai crewai-tools


In [ ]:
# CREWAI — Build a research team
# pip install crewai crewai-tools
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
import os

os.environ["GEMINI_API_KEY"] = os.getenv("GOOGLE_API_KEY", "your-key")

# ── Tools ──
@tool
def search_courses(topic: str) -> str:
    """Search Netsetos course catalog by topic."""
    db = {"genai":"GenAI Complete: 14999 INR, 146hrs, 14 modules, 4.8 stars, 5000+ students",
          "python":"Python Mastery: 9999 INR, 80hrs, 10 modules, 4.7 stars",
          "cloud":"GCP Cloud: 11999 INR, 110hrs, 12 modules, 4.6 stars"}
    return db.get(topic.lower(), "Course not found")

@tool
def get_market_data(topic: str) -> str:
    """Get market trends and salary data for a tech field."""
    data = {"genai":"GenAI engineers: 15-40 LPA. 300% demand growth. Top skill 2025-26.",
            "python":"Python devs: 8-25 LPA. Stable demand. Foundation for ML/AI.",
            "cloud":"Cloud architects: 12-35 LPA. 200% GCP growth."}
    return data.get(topic.lower(), "No market data")

# ── Agents (each with a role, goal, backstory) ──
researcher = Agent(
    role="Course Researcher",
    goal="Find detailed information about Netsetos courses and market demand",
    backstory="You are an expert edtech researcher who analyzes courses and market trends to help students make informed decisions.",
    tools=[search_courses, get_market_data],
    llm="gemini/gemini-2.0-flash",
    verbose=True,
)

analyst = Agent(
    role="Career Analyst",
    goal="Analyze course data and market trends to provide career recommendations",
    backstory="You are a career counselor with 10 years experience in tech hiring. You match courses to career paths.",
    llm="gemini/gemini-2.0-flash",
    verbose=True,
)

writer = Agent(
    role="Report Writer",
    goal="Create a clear, actionable student advisory report",
    backstory="You are a content writer who turns research into easy-to-read, student-friendly reports.",
    llm="gemini/gemini-2.0-flash",
    verbose=True,
)

# ── Tasks (sequential: research → analyze → write) ──
research_task = Task(
    description="Research the GenAI course: details, pricing, market demand, salary trends.",
    expected_output="Detailed course info + market data in structured format.",
    agent=researcher,
)

analysis_task = Task(
    description="Analyze the research. Compare course value vs market opportunity. ROI analysis.",
    expected_output="Career analysis with ROI calculation and recommendation.",
    agent=analyst,
)

report_task = Task(
    description="Write a student advisory report combining research and analysis. Clear, actionable.",
    expected_output="A polished 300-word student advisory report.",
    agent=writer,
)

# ── Crew (execute sequentially) ──
crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[research_task, analysis_task, report_task],
    process=Process.sequential,
    verbose=True,
)

# result = crew.kickoff()
# print(result)

print("CrewAI Research Team:\n")
print("  Agents: Researcher (tools) → Analyst (reasoning) → Writer (compose)")
print("  Process: sequential (each agent gets previous agent's output)")
print("  Researcher uses search_courses + get_market_data")
print("  Analyst interprets data, calculates ROI")
print("  Writer produces polished student advisory report")


> **What just happened?** Three agents, three tasks, one crew. The Researcher uses tools to gather course data and market trends. The Analyst receives that data, interprets it, calculates ROI. The Writer takes both outputs and produces a polished report. Each agent has a role, goal, and backstory that shapes HOW it thinks. The backstory is a system prompt — “10 years in tech hiring” makes the Analyst think like a career counselor.


### Step 2 · Agno — Lightweight Multi-Agent Teams

Agno’s simpler API: Agent + Team + coordinate. Less config, faster prototyping.

**`02_agno_team.py`** · _Block 2 of 3_

AGNO — Lightweight multi-agent teams — pip install agno google-genai


In [ ]:
# AGNO — Lightweight multi-agent teams
# pip install agno google-genai
from agno.agent import Agent
from agno.team import Team
from agno.models.google import Gemini
from agno.tools.function import Function
import os

# ── Tools ──
def search_courses(topic: str) -> str:
    """Search Netsetos courses by topic."""
    db = {"genai":"GenAI: 14999 INR, 146hrs, 4.8 stars", "python":"Python: 9999 INR, 80hrs"}
    return db.get(topic.lower(), "Not found")

def calculate(expression: str) -> str:
    """Evaluate math expression."""
    return str(round(eval(expression, {"__builtins__":{}}), 2))

# ── Agents ──
researcher = Agent(
    name="Researcher",
    role="Find course details and pricing",
    model=Gemini(id="gemini-2.0-flash"),
    tools=[Function(search_courses)],
    instructions=["Always search for course data before answering."],
)

calculator = Agent(
    name="Calculator",
    role="Perform pricing calculations with GST and EMI",
    model=Gemini(id="gemini-2.0-flash"),
    tools=[Function(calculate)],
    instructions=["Show all calculation steps."],
)

advisor = Agent(
    name="Advisor",
    role="Give final course recommendation to student",
    model=Gemini(id="gemini-2.0-flash"),
    instructions=["Be warm, encouraging, and provide clear next steps."],
)

# ── Team ──
team = Team(
    name="Netsetos Advisory",
    agents=[researcher, calculator, advisor],
    instructions=["Research first, calculate pricing, then advise the student."],
    model=Gemini(id="gemini-2.0-flash"),  # coordinator model
)

# response = team.run("I'm interested in GenAI. What's the cost with GST and EMI?")
# print(response.content)

print("Agno Team:\n")
print("  Agents: Researcher (search) + Calculator (math) + Advisor (recommend)")
print("  Team coordinator routes to the right agent")
print("  Simpler API than CrewAI: Agent() + Team() + team.run()")


### Step 3 · Multi-Agent From Scratch — Understand the Pattern

No frameworks. See exactly how agents coordinate, delegate, and combine outputs.

**`03_multi_agent_scratch.py`** · _Block 3 of 3_

MULTI-AGENT FROM SCRATCH — No frameworks


In [ ]:
# MULTI-AGENT FROM SCRATCH — No frameworks
import google.generativeai as genai, json, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class SpecialistAgent:
    def __init__(self, name, role, tools=None):
        self.name, self.role = name, role
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.tools = tools or {}

    def run(self, task, context=""):
        tool_results = []
        for tname, tfunc in self.tools.items():
            tool_results.append(f"{tname}: {json.dumps(tfunc(task))}")
        tool_ctx = "\n".join(tool_results) if tool_results else "No tools used."
        prompt = (f"You are {self.name}, a {self.role}.\n"
                  f"Previous context:\n{context}\n"
                  f"Tool results:\n{tool_ctx}\n"
                  f"Task: {task}\nRespond concisely:")
        return self.model.generate_content(prompt).text.strip()

class AgentTeam:
    def __init__(self, agents):
        self.agents = agents

    def run_sequential(self, task):
        """Each agent gets previous agent's output as context."""
        context, trace = "", []
        for agent in self.agents:
            result = agent.run(task, context)
            trace.append({"agent":agent.name, "output":result[:100]})
            context += f"\n[{agent.name}]: {result}"
        return {"final": result, "trace": trace}

    def run_debate(self, task, rounds=2):
        """Agents debate: each responds to all others' outputs."""
        outputs = {a.name: "" for a in self.agents}
        for rnd in range(rounds):
            for agent in self.agents:
                others = "\n".join(f"[{n}]: {o}" for n,o in outputs.items() if n!=agent.name and o)
                outputs[agent.name] = agent.run(task, f"Round {rnd+1}. Others said:\n{others}")
        # Synthesize
        model = genai.GenerativeModel("gemini-2.0-flash")
        all_views = "\n".join(f"[{n}]: {o}" for n,o in outputs.items())
        final = model.generate_content(f"Synthesize these expert views:\n{all_views}\nTask: {task}").text
        return {"final": final.strip(), "views": outputs}

# ── Build team ──
def search(q): return {"genai":"14999 INR, 146hrs"}.get(q.split()[0].lower(),"?")

researcher = SpecialistAgent("Researcher", "edtech researcher", {"search": search})
analyst = SpecialistAgent("Analyst", "career analyst with 10 years hiring experience")
writer = SpecialistAgent("Writer", "student-friendly content writer")

team = AgentTeam([researcher, analyst, writer])

print("Sequential Team:\n")
r = team.run_sequential("Should a student invest in the GenAI course?")
for t in r["trace"]: print(f"  [{t['agent']}]: {t['output'][:80]}")

print(f"\nDebate Team:\n")
optimist = SpecialistAgent("Optimist", "always sees opportunity and upside")
skeptic = SpecialistAgent("Skeptic", "questions assumptions, finds risks")
r2 = AgentTeam([optimist, skeptic]).run_debate("Is the GenAI course worth 14999?")
print(f"  Optimist: {r2['views']['Optimist'][:80]}")
print(f"  Skeptic: {r2['views']['Skeptic'][:80]}")
print(f"  Synthesis: {r2['final'][:100]}")


> **What just happened?** Two patterns in one script. Sequential: Researcher → Analyst → Writer, each building on the previous output. Debate: Optimist and Skeptic argue for 2 rounds, then a synthesizer combines their views. The debate pattern is remarkably effective: by forcing agents to challenge each other, the final answer is more nuanced than any single agent could produce.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
